# VizEx — build the file index and master trial table

Everything here is derived by scanning `005_cleaned_data/` and parsing filenames.
No previous index is read, so the tables always reflect the current, correctly
anonymised files on disk.

**Outputs written to `015_tables/`**

| file | grain | purpose |
|---|---|---|
| `file_index.csv` | one row per file | complete inventory, provenance |
| `trials_master.csv` | one row per trial | **the master table for the analysis** |
| `targets_master.csv` | one row per target image | the images shown on validation trials |

`trials_master.csv` carries the `_cropped.png` filename in `png_filename`, because
that is the only file the analysis reads. The other file types are recorded
alongside it for provenance but are never analysed.

Run this first, then `015`, `030`, `035`.

In [1]:
## imports

import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
## paths and configuration

PROJECT_PATH = Path().resolve().parent
TABLES_PATH = PROJECT_PATH / "015_tables"
TRIAL_IMG_DIR = PROJECT_PATH / "005_cleaned_data"
TARGET_IMG_DIR = PROJECT_PATH / "007_targets"

TABLES_PATH.mkdir(parents=True, exist_ok=True)

FILE_INDEX_CSV = TABLES_PATH / "file_index.csv"
TRIALS_MASTER_CSV = TABLES_PATH / "trials_master.csv"
TARGETS_MASTER_CSV = TABLES_PATH / "targets_master.csv"

# Optional. If present, a CSV with columns participant_code, trialn and an optional
# `reason`, marking trials to drop by hand. Absence is fine and is reported below.
MANUAL_EXCLUSIONS_CSV = TABLES_PATH / "manual_exclusions.csv"

# Strobe condition codes -> flicker frequency in Hz
HZ_MAP = {1: 2.5, 2: 5, 3: 10, 4: 15, 5: 20, 6: 40, 7: 80}
CONTROL_CODE = "7"

# Condition codes that are not analysis trials at all
NON_TRIAL_CODES = {"9", "POSTQ"}

# The file type whose image the analysis actually reads
ANALYSIS_FILE_TYPE = "cropped"

# Which file types each complete trial should have. None = infer from the data,
# which is the safe default if the export format has changed.
EXPECTED_FILE_TYPES = None

In [3]:
## parse every file in the cleaned data directory

FILENAME_RE = re.compile(
    r"^saveTime_(?P<save_time>\d+)"
    r"__sessionTime_(?P<session_time>\d+)"
    r"__participant_(?P<participant_code>[^_]+)"
    r"__trialN_(?P<trialn>\d+)"
    r"__trialT_(?P<condition_code>[^_]+)"
    r"__fileType_(?P<file_type>[^_.]+)"
    r"\.(?P<ext>[A-Za-z0-9]+)$"
)

if not TRIAL_IMG_DIR.is_dir():
    raise NotADirectoryError(f"cleaned data directory not found: {TRIAL_IMG_DIR}")

all_files = sorted(p for p in TRIAL_IMG_DIR.rglob("*") if p.is_file())

parsed, unparseable = [], []
for path in all_files:
    m = FILENAME_RE.match(path.name)
    if not m:
        unparseable.append(path.name)
        continue
    d = m.groupdict()
    parsed.append({
        "filename": path.name,
        "participant_code": d["participant_code"],
        "session_time": d["session_time"],
        "save_time": d["save_time"],
        "trialn": int(d["trialn"]),
        "condition_code": d["condition_code"],
        "file_type": d["file_type"],
        "ext": d["ext"],
        "size_bytes": path.stat().st_size,
    })

file_index = pd.DataFrame(parsed)

print(f"scanned {len(all_files):,} files in {TRIAL_IMG_DIR}")
print(f"  parsed:      {len(file_index):,}")
print(f"  unparseable: {len(unparseable):,}")
if unparseable:
    print("\n  files whose names do not match the expected pattern:")
    for name in unparseable[:20]:
        print(f"    {name}")
    if len(unparseable) > 20:
        print(f"    ... and {len(unparseable) - 20} more")

if file_index.empty:
    raise ValueError(f"No parseable files found in {TRIAL_IMG_DIR}.")

scanned 1,581 files in C:\Users\trevo\research\vizex_replication\005_cleaned_data
  parsed:      1,581
  unparseable: 0


In [4]:
## file types present, and duplicate detection

type_counts = file_index["file_type"].value_counts()
print("file types found:")
for ftype, n in type_counts.items():
    ext = sorted(file_index.loc[file_index["file_type"] == ftype, "ext"].unique())
    print(f"  {ftype:<12} {n:>6,}  (.{'/.'.join(ext)})")

file_types = (sorted(type_counts.index) if EXPECTED_FILE_TYPES is None
              else list(EXPECTED_FILE_TYPES))
print(f"\nexpecting {len(file_types)} file(s) per trial: {file_types}"
      f"{'  [inferred from the data]' if EXPECTED_FILE_TYPES is None else '  [set explicitly]'}")

if ANALYSIS_FILE_TYPE not in file_types:
    raise ValueError(
        f"No '{ANALYSIS_FILE_TYPE}' files found. The analysis reads only this file "
        f"type, so there is nothing to analyse."
    )

# A trial is identified by participant + trial number. Each file type should appear once.
TRIAL_KEY = ["participant_code", "trialn"]
dupes = file_index[file_index.duplicated(TRIAL_KEY + ["file_type"], keep=False)]
if not dupes.empty:
    print(f"\nWARNING: {len(dupes)} files share a participant/trial/file-type key:")
    print(dupes.sort_values(TRIAL_KEY + ["file_type"])
               [TRIAL_KEY + ["file_type", "filename"]].to_string(index=False))
    raise ValueError("Duplicate files for the same trial and file type -- resolve before continuing.")

file_index = file_index.sort_values(TRIAL_KEY + ["file_type"]).reset_index(drop=True)
file_index.to_csv(FILE_INDEX_CSV, index=False)
print(f"\nSaved {FILE_INDEX_CSV.name}: {len(file_index):,} rows")

file types found:
  cropped       1,581  (.png)

expecting 1 file(s) per trial: ['cropped']  [inferred from the data]

Saved file_index.csv: 1,581 rows


In [5]:
## collapse to one row per trial

def format_hz(hz) -> str:
    """5.0 -> '5', 2.5 -> '2.5'."""
    hz = float(hz)
    return str(int(hz)) if hz.is_integer() else str(hz)


def condition_code_to_int(x):
    """Integer strobe code, or None for validation image codes like 'C14'."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    try:
        f = float(str(x).strip())
    except ValueError:
        return None
    return int(f) if np.isfinite(f) and float(f).is_integer() else None


rows = []
for (participant, trialn), grp in file_index.groupby(TRIAL_KEY, sort=True):
    by_type = dict(zip(grp["file_type"], grp["filename"]))

    # These should be constant within a trial; report if they are not.
    codes = sorted(grp["condition_code"].unique())
    sessions = sorted(grp["session_time"].unique())
    if len(codes) > 1 or len(sessions) > 1:
        print(f"WARNING: {participant} trial {trialn} has inconsistent metadata "
              f"across its files -- condition {codes}, session {sessions}")

    code = codes[0]
    ci = condition_code_to_int(code)

    row = {
        "participant_code": participant,
        "session_time": sessions[0],
        "save_time": grp["save_time"].iloc[0],
        "trialn": trialn,
        "condition_code": code,
        "png_filename": by_type.get(ANALYSIS_FILE_TYPE),
        "n_files": len(by_type),
        "missing_file_types": ",".join(t for t in file_types if t not in by_type),
    }
    for ftype in file_types:
        row[f"file_{ftype}"] = by_type.get(ftype)

    if code in NON_TRIAL_CODES:
        row["condition_type"] = "NOT_A_TRIAL"
        row["stimulus_type"] = pd.NA
        row["condition_hz"] = pd.NA
        row["condition_name"] = pd.NA
    elif ci is None:
        row["condition_type"] = "VALIDATION"      # a target image was shown
        row["stimulus_type"] = "IMAGE"
        row["condition_hz"] = pd.NA
        row["condition_name"] = pd.NA
    elif ci in HZ_MAP:
        row["condition_type"] = "CONTROL" if code == CONTROL_CODE else "EXPERIMENTAL"
        row["stimulus_type"] = "LIGHT"
        row["condition_hz"] = HZ_MAP[ci]
        row["condition_name"] = f"{format_hz(HZ_MAP[ci])} Hz"
    else:
        raise ValueError(
            f"{participant} trial {trialn}: unrecognised condition code {code!r}. "
            f"Add it to HZ_MAP or NON_TRIAL_CODES."
        )

    rows.append(row)

trials = pd.DataFrame(rows)
trials["condition_hz"] = pd.array(trials["condition_hz"], dtype="Float64")

print(f"{len(trials):,} trials across {trials['participant_code'].nunique()} participants")

1,581 trials across 99 participants


In [6]:
## completeness report: missing files and missing trials

print("=" * 72)
print("COMPLETENESS REPORT")
print("=" * 72)

# --- trials missing one or more expected files ---
incomplete = trials[trials["missing_file_types"] != ""]
print(f"\n1. Trials missing at least one expected file: {len(incomplete)} of {len(trials)}")
if not incomplete.empty:
    print(f"   (a complete trial has: {', '.join(file_types)})")
    for missing_set, grp in incomplete.groupby("missing_file_types"):
        print(f"\n   missing [{missing_set}] -- {len(grp)} trial(s):")
        for _, r in grp.head(15).iterrows():
            print(f"     {r['participant_code']}  trial {r['trialn']:>3}  cond {r['condition_code']}")
        if len(grp) > 15:
            print(f"     ... and {len(grp) - 15} more")

# --- trials with no analysable image ---
no_image = trials[trials["png_filename"].isna()]
print(f"\n2. Trials with no {ANALYSIS_FILE_TYPE} image (cannot be analysed): {len(no_image)}")
for _, r in no_image.iterrows():
    print(f"     {r['participant_code']}  trial {r['trialn']:>3}  cond {r['condition_code']}")

# --- gaps in the trial sequence per participant ---
print("\n3. Gaps in the trial number sequence, per participant:")
gap_rows = []
for participant, grp in trials.groupby("participant_code"):
    present = set(grp["trialn"])
    expected = set(range(min(present), max(present) + 1))
    gaps = sorted(expected - present)
    if gaps:
        gap_rows.append((participant, gaps))
        print(f"     {participant}: missing trial number(s) {gaps} "
              f"(range {min(present)}-{max(present)})")
if not gap_rows:
    print("     none -- every participant's trial numbers are contiguous")

# --- participants whose set of conditions differs from the modal set ---
print("\n4. Condition coverage, per participant:")
analysis_trials = trials[~trials["condition_code"].isin(NON_TRIAL_CODES)]
cond_sets = analysis_trials.groupby("participant_code")["condition_code"].apply(frozenset)
modal_set = Counter(cond_sets).most_common(1)[0][0] if len(cond_sets) else frozenset()
print(f"     modal condition set has {len(modal_set)} conditions "
      f"({Counter(cond_sets).most_common(1)[0][1]} participants match it)")
for participant, s in cond_sets.items():
    if s != modal_set:
        extra, absent = sorted(s - modal_set), sorted(modal_set - s)
        bits = []
        if absent:
            bits.append(f"missing {absent}")
        if extra:
            bits.append(f"extra {extra}")
        print(f"     {participant}: {'; '.join(bits)}")

# --- counts per condition ---
print("\n5. Trials per condition:")
counts = (analysis_trials.groupby(["condition_type", "condition_code"])
          .size().rename("n").reset_index())
for line in counts.to_string(index=False).split("\n"):
    print("     " + line)

print("\n" + "=" * 72)

COMPLETENESS REPORT

1. Trials missing at least one expected file: 0 of 1581

2. Trials with no cropped image (cannot be analysed): 0

3. Gaps in the trial number sequence, per participant:
     P306: missing trial number(s) [4, 6, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100] (range 2-102)
     P309: missing trial number(s) [12] (range 2-15)
     P324: missing trial number(s) [3, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 1

In [7]:
## apply exclusions and write the master trial table

trials["exclusion_reason"] = ""

is_non_trial = trials["condition_code"].isin(NON_TRIAL_CODES)
trials.loc[is_non_trial, "exclusion_reason"] = "not_a_trial"

no_img = trials["png_filename"].isna() & (trials["exclusion_reason"] == "")
trials.loc[no_img, "exclusion_reason"] = f"no_{ANALYSIS_FILE_TYPE}_image"

# Optional hand-curated exclusions. The archived index carried a `keep_manual`
# column; regenerating from filenames cannot recover it, so any manual decisions
# must live in this file.
if MANUAL_EXCLUSIONS_CSV.exists():
    manual = pd.read_csv(MANUAL_EXCLUSIONS_CSV)
    manual["trialn"] = manual["trialn"].astype(int)
    keys = set(zip(manual["participant_code"].astype(str), manual["trialn"]))
    hit = [
        (p, t) in keys and r == ""
        for p, t, r in zip(trials["participant_code"].astype(str),
                           trials["trialn"], trials["exclusion_reason"])
    ]
    trials.loc[hit, "exclusion_reason"] = "manual"
    print(f"Applied {sum(hit)} manual exclusion(s) from {MANUAL_EXCLUSIONS_CSV.name}")
else:
    print(f"No {MANUAL_EXCLUSIONS_CSV.name} found -- no manual exclusions applied.")
    print("  If the archived index had hand-set keep flags, recreate them in that file")
    print("  with columns: participant_code, trialn, reason")

trials["in_analysis"] = trials["exclusion_reason"] == ""

column_order = (
    ["participant_code", "session_time", "save_time", "trialn",
     "condition_code", "condition_name", "condition_hz", "condition_type",
     "stimulus_type", "in_analysis", "exclusion_reason",
     "png_filename", "n_files", "missing_file_types"]
    + [f"file_{t}" for t in file_types]
)
trials = trials[column_order].sort_values(TRIAL_KEY).reset_index(drop=True)

trials.to_csv(TRIALS_MASTER_CSV, index=False)

print(f"\nSaved {TRIALS_MASTER_CSV.name}: {len(trials):,} trials")
print(f"  in analysis: {int(trials['in_analysis'].sum()):,}")
print("  excluded:")
excl = trials.loc[~trials["in_analysis"], "exclusion_reason"].value_counts()
for reason, n in excl.items():
    print(f"    {reason:<24} {n:>5}")
print("\n  breakdown of the analysis set:")
print(trials[trials["in_analysis"]].groupby(["stimulus_type", "condition_type"])
      .size().to_string().replace("\n", "\n    "))

display(trials.head())

No manual_exclusions.csv found -- no manual exclusions applied.
  If the archived index had hand-set keep flags, recreate them in that file
  with columns: participant_code, trialn, reason

Saved trials_master.csv: 1,581 trials
  in analysis: 1,581
  excluded:

  breakdown of the analysis set:
stimulus_type  condition_type
    IMAGE          VALIDATION         296
    LIGHT          CONTROL             98
                   EXPERIMENTAL      1187


,participant_code,session_time,save_time,trialn,condition_code,condition_name,condition_hz,condition_type,stimulus_type,in_analysis,exclusion_reason,png_filename,n_files,missing_file_types,file_cropped
0,P301,202510170920,2510171012,2,7,80 Hz,80.0,CONTROL,LIGHT,True,,saveTime_2510171012__sessionTime_202510170920_...,1,,saveTime_2510171012__sessionTime_202510170920_...
1,P301,202510170920,2510171017,3,2,5 Hz,5.0,EXPERIMENTAL,LIGHT,True,,saveTime_2510171017__sessionTime_202510170920_...,1,,saveTime_2510171017__sessionTime_202510170920_...
2,P301,202510170920,2510171020,4,B09,NaN,<NA>,VALIDATION,IMAGE,True,,saveTime_2510171020__sessionTime_202510170920_...,1,,saveTime_2510171020__sessionTime_202510170920_...
3,P301,202510170920,2510171023,5,2,5 Hz,5.0,EXPERIMENTAL,LIGHT,True,,saveTime_2510171023__sessionTime_202510170920_...,1,,saveTime_2510171023__sessionTime_202510170920_...
4,P301,202510170920,2510171038,6,C00,NaN,<NA>,VALIDATION,IMAGE,True,,saveTime_2510171038__sessionTime_202510170920_...,1,,saveTime_2510171038__sessionTime_202510170920_...


In [8]:
## target images: A00-A14, B00-B14, C00-C14

targets = pd.DataFrame([
    {"condition_code": f"{p}{i:02d}",
     "condition_name": f"Image {p}{i:02d}",
     "png_filename": f"{p}{i:02d}.png"}
    for p in ("A", "B", "C") for i in range(15)
])

targets["exists"] = [(TARGET_IMG_DIR / f).exists() for f in targets["png_filename"]]

missing_targets = targets.loc[~targets["exists"], "condition_code"].tolist()
if missing_targets:
    print(f"WARNING: {len(missing_targets)} target image(s) not found in {TARGET_IMG_DIR}:")
    print(f"  {missing_targets}")

# Validation trials can only be modelled against a target that exists.
used = set(trials.loc[trials["condition_type"] == "VALIDATION", "condition_code"])
unusable = sorted(used & set(missing_targets))
if unusable:
    print(f"WARNING: {len(unusable)} validation condition(s) were shown to participants "
          f"but have no target image: {unusable}")

orphan = sorted(set(targets.loc[targets['exists'], 'condition_code']) - used)
if orphan:
    print(f"Note: {len(orphan)} target image(s) exist but were never shown: {orphan}")

targets.to_csv(TARGETS_MASTER_CSV, index=False)
print(f"\nSaved {TARGETS_MASTER_CSV.name}: {len(targets)} rows "
      f"({int(targets['exists'].sum())} present on disk)")
display(targets.head())


Saved targets_master.csv: 45 rows (45 present on disk)


,condition_code,condition_name,png_filename,exists
0,A00,Image A00,A00.png,True
1,A01,Image A01,A01.png,True
2,A02,Image A02,A02.png,True
3,A03,Image A03,A03.png,True
4,A04,Image A04,A04.png,True
